In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Notebook setup works")

Notebook setup works


In [12]:
# load dataset
df = pd.read_csv("../data/raw/shot_logs.csv")

# basic info
print(df.shape)
df.head()

# check for missing values
df.columns
df.info()
df.isna().sum().sort_values(ascending=False)

(128069, 21)
<class 'pandas.DataFrame'>
RangeIndex: 128069 entries, 0 to 128068
Data columns (total 21 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   GAME_ID                     128069 non-null  int64  
 1   MATCHUP                     128069 non-null  str    
 2   LOCATION                    128069 non-null  str    
 3   W                           128069 non-null  str    
 4   FINAL_MARGIN                128069 non-null  int64  
 5   SHOT_NUMBER                 128069 non-null  int64  
 6   PERIOD                      128069 non-null  int64  
 7   GAME_CLOCK                  128069 non-null  str    
 8   SHOT_CLOCK                  122502 non-null  float64
 9   DRIBBLES                    128069 non-null  int64  
 10  TOUCH_TIME                  128069 non-null  float64
 11  SHOT_DIST                   128069 non-null  float64
 12  PTS_TYPE                    128069 non-null  int64  
 13  SHOT_RESULT 

SHOT_CLOCK                    5567
MATCHUP                          0
LOCATION                         0
W                                0
GAME_ID                          0
FINAL_MARGIN                     0
SHOT_NUMBER                      0
PERIOD                           0
GAME_CLOCK                       0
DRIBBLES                         0
TOUCH_TIME                       0
SHOT_DIST                        0
PTS_TYPE                         0
SHOT_RESULT                      0
CLOSEST_DEFENDER                 0
CLOSEST_DEFENDER_PLAYER_ID       0
CLOSE_DEF_DIST                   0
FGM                              0
PTS                              0
player_name                      0
player_id                        0
dtype: int64

In [13]:
# create working copy and standardize column names
df_clean = df.copy()
df_clean.columns = [c.lower().strip() for c in df_clean.columns]

# define binary target (1 = made, 0 = missed)
df_clean["made"] = df_clean["fgm"]

# convert GAME_CLOCK (MM:SS) to numeric seconds
def clock_to_seconds(x):
    if pd.isna(x):
        return np.nan
    mins, secs = x.split(":")
    return int(mins) * 60 + int(secs)

df_clean["game_clock_sec"] = df_clean["game_clock"].apply(clock_to_seconds)

# create missing indicator and fill missing shot clock with median
df_clean["shot_clock_missing"] = df_clean["shot_clock"].isna().astype(int)
df_clean["shot_clock"] = df_clean["shot_clock"].fillna(df_clean["shot_clock"].median())

# convert relevant columns to numeric types
num_cols = [
    "shot_clock",
    "dribbles",
    "touch_time",
    "shot_dist",
    "close_def_dist",
    "final_margin",
]

for col in num_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# filter out impossible or invalid values
df_clean = df_clean[(df_clean["shot_clock"] >= 0) & (df_clean["shot_clock"] <= 24)]
df_clean = df_clean[df_clean["shot_dist"] >= 0]
df_clean = df_clean[df_clean["close_def_dist"] >= 0]
df_clean = df_clean[df_clean["touch_time"] >= 0]
df_clean = df_clean[df_clean["dribbles"] >= 0]

# remove any rows missing the target variable
df_clean = df_clean.dropna(subset=["made"])

# keep only relevant features for modeling
df_clean = df_clean[
    [
        "player_name",
        "matchup",
        "location",
        "period",
        "game_clock_sec",
        "shot_clock",
        "shot_clock_missing",
        "dribbles",
        "touch_time",
        "shot_dist",
        "close_def_dist",
        "pts_type",
        "made",
    ]
]

# check remaining missing values
df_clean.isna().sum()

player_name           0
matchup               0
location              0
period                0
game_clock_sec        0
shot_clock            0
shot_clock_missing    0
dribbles              0
touch_time            0
shot_dist             0
close_def_dist        0
pts_type              0
made                  0
dtype: int64

In [ ]:
# 1. Shot clock as a fraction of the 24-second shot clock;
#    zero out rows where shot_clock was originally missing
df_clean["shot_clock_pct"] = df_clean["shot_clock"] / 24
df_clean.loc[df_clean["shot_clock_missing"] == 1, "shot_clock_pct"] = 0


# 2. Game clock as a fraction of the period's total time (720 seconds per quarter)
df_clean["game_clock_pct"] = df_clean["game_clock_sec"] / (df_clean["period"] * 720)


# 3. Catch-and-shoot flag: no dribbles and minimal time with the ball
df_clean["catch_and_shoot"] = (
    (df_clean["dribbles"] == 0) & (df_clean["touch_time"] < 2)
).astype(int)


# 4. Dribble pull-up flag: heavy ball-handling before the shot
df_clean["dribble_pull_up"] = (
    (df_clean["dribbles"] >= 3) & (df_clean["touch_time"] > 2)
).astype(int)


# 5. Interaction term: defender proximity × shot distance
df_clean["def_dist_x_shot_dist"] = df_clean["close_def_dist"] * df_clean["shot_dist"]


# 6. Quadratic shot distance to capture non-linear range penalty
df_clean["shot_dist_squared"] = df_clean["shot_dist"] ** 2


print("Features engineered. New columns:")
print([c for c in df_clean.columns if c not in [
    "player_name", "matchup", "location", "period", "game_clock_sec",
    "shot_clock", "shot_clock_missing", "dribbles", "touch_time",
    "shot_dist", "close_def_dist", "pts_type", "made"
]])

# save cleaned dataset with all engineered features
df_clean.to_csv("../data/processed/shot_logs_cleaned.csv", index=False)
print("Saved to data/processed/shot_logs_cleaned.csv")

In [ ]:
# All modeling columns — excludes identifiers (player_name, matchup) and raw target (made)
features = [
    "location",
    "period",
    "game_clock_sec",
    "shot_clock",
    "shot_clock_missing",
    "dribbles",
    "touch_time",
    "shot_dist",
    "close_def_dist",
    "pts_type",
    # engineered features
    "shot_clock_pct",
    "game_clock_pct",
    "catch_and_shoot",
    "dribble_pull_up",
    "def_dist_x_shot_dist",
    "shot_dist_squared",
]

target = "made"

# Sanity checks
print("Feature matrix shape:", df_clean[features].shape)
print()
print("Target distribution:")
print(df_clean[target].value_counts())